# 22 — Selective future distillation: a falsifiable next study

This tutorial develops the most promising research extension while keeping
its implementation status explicit. The cached comparison is real; the
calculations below are **synthetic teaching examples**. No video teacher is
loaded and no real skeleton student is trained here. The goal is to define
a method whose assumptions can be challenged before an expensive run.

Let H be all permitted student history and C a declared function of H
containing current state and observation quality. For a teacher target Y,
define population temporal innovation as
$u(H)=E[Y\mid H]-E[Y\mid C]$.
This is information accessible to the student beyond its current reference.
It is different from the residual of a predictor that can see RGB.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display, SVG, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/gavd6_sjepa').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
EVIDENCE = ROOT / 'work/artifacts/iclr-bridge-2026-09-11'
PANEL = ROOT / 'outputs/iclr-bridge-cached-20260911'
def read_json(path):
    return json.loads(Path(path).read_text())
pd.set_option('display.precision', 8)
print('Repository:', ROOT)
print('Execution: read-only evidence inspection + labeled synthetic calculations')

In [ ]:
display(SVG(filename=str(ROOT / 'docs/studies/iclr/figures/05_matched_pose_futures.svg')))

## 1. Derive the quantity before constructing an estimator

Under finite second moments, common squared-error units, and C measurable
from H, the tower property gives $E[u\mid C]=0$. Orthogonality gives
$E\|Y-E[Y\mid C]\|^2-E\|Y-E[Y\mid H]\|^2=E\|u\|^2$.
This is standard conditional-expectation algebra, not a new theorem.
A finite ridge family estimates neither expectation exactly. Its held-source
performance difference is a model comparison, not conditional mutual
information or proof that the entire skeleton contains no extra signal.

In practice, target units, dimension weights and source weights must remain
common when comparing predictors. Fold-specific target normalization is
appropriate for the declared score but must be inverted before combining
residual arrays from different subfits in one learned target basis.

## 2. Show why RGB complementarity is neither necessary nor sufficient

We construct independent signs R and S with all four combinations equally
likely. A “student” observes S and an “RGB reference” observes R. In the
shared-signal case RGB also contains S and the target equals S: adding S
after RGB gives zero improvement, yet the student predicts perfectly.
In the interaction case Y=R×S: the joint observations determine Y, but S
alone has conditional mean zero. This second example concerns rich
predictors that can represent the interaction; the current linear panel
does not claim to estimate it.

In [ ]:
# Exact finite probability space; no fitted GAVD model is involved.
R=np.array([-1.,-1.,1.,1.]); S=np.array([-1.,1.,-1.,1.])
def mse(y,p): return float(np.mean((y-p)**2))
shared=S; interaction=R*S
examples=pd.DataFrame([
    {'example':'Shared student signal', 'RGB_reference_MSE':mse(shared,S),
     'joint_MSE':0., 'student_only_MSE':mse(shared,S)},
    {'example':'Pure interaction', 'RGB_reference_MSE':mse(interaction,np.zeros(4)),
     'joint_MSE':0., 'student_only_MSE':mse(interaction,np.zeros(4))},
])
examples['gain_after_RGB']=examples.RGB_reference_MSE-examples.joint_MSE
display(examples)
assert examples.loc[0,'gain_after_RGB']==0 and examples.loc[0,'student_only_MSE']==0
assert examples.loc[1,'gain_after_RGB']==1 and examples.loc[1,'student_only_MSE']==1

## 3. Build a genuine paired teacher target

Reflection-even and reflection-odd pooled components are (Y+Y_M)/2 and
(Y−Y_M)/2. We must actually encode both original and mirrored RGB videos
using the same model, temporal boundaries and projection. If spatial tokens
are used, their positions need a declared alignment. Mirroring a pose
array does not produce Y_M. The helper below validates array arithmetic,
identities and an explicit shared basis; it does not verify external video
files or invent missing teacher evidence.

A teacher's odd component might reflect camera layout, clothing or readable
text rather than anatomical motion. Laterality probes and nuisance controls
are therefore needed even when the decomposition is exact.

In [ ]:
from gavd6_sjepa.research_directions.iclr_bridge.symmetry_calibration import paired_teacher_components
rng=np.random.default_rng(2201)
original=rng.normal(size=(8,6)); mirrored=rng.normal(size=(8,6))
pair=paired_teacher_components(original,mirrored,window_ids=[f'toy_{i}' for i in range(8)],
    original_evidence_id='synthetic_original_2201',mirrored_evidence_id='synthetic_mirror_2201',
    shared_basis_id='synthetic_basis_6',evidence_kind='synthetic_teaching')
np.testing.assert_allclose(pair['even']+pair['odd'],original)
np.testing.assert_allclose(pair['even']-pair['odd'],mirrored)
display(pd.DataFrame([pair['provenance']]))

## 4. Select directions for temporal accessibility, not merely large variance

The proposed method learns teacher directions using only current training
sources. Within that partition, fit current-state and history predictors,
construct candidate directions from cross-fitted raw-unit predictions or
a jointly regularized model, and evaluate their extra predictable error
reduction on source-held inner data. Include rank zero, meaning no transfer.
Refit the chosen construction using outer-training sources and test it
once on outer sources. Candidate ranks, penalties and target families must
be finite and fixed before fitting. Do not rank directions by outer scores.

Large teacher variance is insufficient: a background-sensitive direction
can vary strongly while being unpredictable from a skeleton. Conversely,
a low-variance directional-motion feature may be useful. The following
exact synthetic table separates those properties. It illustrates the
selection criterion; it does not implement a validated real target selector.

In [ ]:
# Enumerate all independent signs: appearance A, current state C and history V.
import itertools
A,C,V=np.asarray(list(itertools.product([-1.,1.],repeat=3))).T
Y=np.column_stack([10*A, C, .5*V])
pred_current=np.column_stack([np.zeros(8),C,np.zeros(8)])
pred_history=np.column_stack([np.zeros(8),C,.5*V])
gain=np.mean((Y-pred_current)**2,axis=0)-np.mean((Y-pred_history)**2,axis=0)
display(pd.DataFrame({'teacher_direction':['appearance','current posture','history velocity'],
    'variance':Y.var(axis=0),'extra_predictable_MSE_reduction':gain}))
selected=np.flatnonzero(gain>0)
null_selected=np.flatnonzero(np.zeros_like(gain)>0)
print('Oracle teaching selection:',selected.tolist(),'rank-zero control:',null_selected.tolist())
assert selected.tolist()==[2] and len(null_selected)==0

## 5. Make the distillation comparison capable of disproving the idea

The actual student experiment must compare the same encoder, training
sources, optimization budget and evaluation decoder under: ordinary S-JEPA,
full-feature video distillation, selected-target distillation, rank-matched
random or variance-selected targets, and no transfer. Include matched
initialization and direct kinematics. If a student simply copies an
already sufficient handcrafted forecast, it has not demonstrated added
teacher knowledge. A common independent future-motion endpoint is needed;
lower loss on a smaller or easier selected target is not a fair success metric.

Use a future block encoded without the observed prefix when claiming
future-only supervision. Set horizons in physical time and audit each
encoder's temporal receptive field. Preserve both an anatomical time-even
probe and a time-odd probe. Test camera/view changes, missing joints and
matched current posture. Keep all augmented or paired clips from one source
on the same side of every split; participant separation requires identities
or a dataset that supplies them.

A minimal study first tests one horizon, one frozen teacher layer, one
candidate family and one independent endpoint. Expand only after this
bounded comparison reveals an interpretable effect. Current caches lack
future pose and transformed video encodings, so these real experiments
remain explicit next steps rather than disabled cells masquerading as results.

In [ ]:
display(SVG(filename=str(ROOT / 'docs/studies/iclr/figures/04_research_workflow.svg')))

## 6. Position the contribution honestly

Future privileged supervision and spectral target selection already exist:
see [Overlooked Poses](https://arxiv.org/abs/2208.01302) and
[Spectral-guided Physical Dynamics Distillation](https://openreview.net/forum?id=P6F4MxtOKp).
Split invariant/equivariant representations and skeleton masked-feature
prediction also have close precedents. The defensible opportunity is a
tested selection rule for **extra student-accessible temporal information
beyond current state/support**, an explicit no-transfer outcome, and
verified improvement on independent motion observables. Combining named
losses alone would be a weak contribution.

The paper draft reports current evidence and identifies missing experiments.
A strong main-track claim still needs useful student transfer, comparison
with close alternatives, and confirmation on independent sources or people.
The absence of a positive result is a reason to sharpen the experiment,
not to convert synthetic calibration into scientific evidence.